# OME_ZARR explorer

Zarr files are pyramid structures that allow easy access to large imaging datasets. High Content, high resolution images can reach the 100TBs easily. One of the main problems of imaging data, apart from storage is that processing and displaying images at high resolution is resource intensive. This is to the most level solved by using the file Zarr structure, leading to OME-Zarrs and what we call OME-NGFF.

If wou wanna know more about this file structure and how images are stored, you can go here:
- Zarr website
- Fractal
- OME-NGFF

I have also further explained in the README file in section

-----

### Packages that allow access to OME-Zarr files

One requires more preparation than for opening tiff files or png, since the images are stored in binary files hidden at the bottom of the pyramid. 

There are a few packages that facilitate this:
- NGIO: 
- EZ-Zarr
- Napari

In this course we will look at all of this, so that you feel comfortable with opening zarr formatted imaging data.

There are more information on the README file

---------

## EZ-Zarr


In [1]:
import numpy

In [2]:
# Loading the neccesary packages
from ez_zarr import ome_zarr, plotting, utils

In [ ]:
# Setting paths
zarr_path = "20200812-CardiomyocyteDifferentiation14-Cycle1_mip.zarr"
image_path = "20200812-CardiomyocyteDifferentiation14-Cycle1_mip.zarr/B/03/0"

In [ ]:
# Loading a plate
plateL = ome_zarr.import_plate(zarr_path) # returns the plate list - list of images
plateL
plateL.get_layout()

In [ ]:
# Loading a single well image

imageA = ome_zarr.Image('image_path')

In [ ]:
# Checking well properties

In [ ]:
# Showing an image on matplotlib - correct brightness, merged channels, each channel on their own

In [ ]:
# Add segmentation masks to plotting, try plot only the area of one masks

In [ ]:
# Extra - if you have too much time or are interested

## Feature Extraction Table - access location and plot feature extraction numbers such as area onto your image

## NGIO: 

This packages is 

In [ ]:
from pathlib import Path

from ngio import open_ome_zarr_plate, open_ome_zarr_well,open_ome_zarr_container
from ngio_helpers import ZarrWellIterator, Formatter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ngio import open_ome_zarr_well, open_image

In [ ]:
plates = {}
plates["Day8_IFN_10x_Maximum.zarr"] = Path("/cluster/work/liberali/zarr-data/maaraujo/fractal/20260515_IBD_stain_test/Day8_IFN_10x_Maximum.zarr")
plates["Day8_Normal_10x_Maximum.zarr"] = Path("/cluster/work/liberali/zarr-data/maaraujo/fractal/20260515_IBD_stain_test/Day8_Normal_10x_Maximum.zarr")

In [ ]:

well_path =  Path(plates["Day8_Normal_10x_Maximum.zarr"]/"E/02")   # <- your well
level      = "0"                       # "0" = full resolution
out_dir    = Path("./output_for_poster")
use_window = True                      # False -> percentile stretch

row, col  = well_path.parent.name, well_path.name
well_name = f"{well_path.parent.parent.stem}_{row}{col}"

# --- load the whole well -----------------------------------------------------
well  = open_ome_zarr_well(well_path, mode="r")
image = open_image(well_path / well.paths()[0], path=level, mode="r")
print(image.dimensions, image.meta.paths, image.channel_labels)

axes = [a for a in ("t", "c", "z", "y", "x") if image.has_axis(a)]
arr  = image.get_array(axes_order=axes, mode="dask")

if "t" in axes:                                   # first timepoint
    arr = arr[0]; axes.remove("t")
if "z" in axes:                                   # max projection
    arr = arr.max(axis=axes.index("z")); axes.remove("z")
if "c" not in axes:
    arr = arr[None]
planes = arr.compute().astype(np.float32)         # -> (c, y, x)

# --- channel metadata --------------------------------------------------------
channels = image.channels_meta.channels

def hex_to_rgb(h):
    h = h.lstrip("#")
    return np.array([int(h[i:i + 2], 16) for i in (0, 2, 4)]) / 255

def normalise(plane, ch):
    vis = ch.channel_visualisation
    if use_window and vis.end > vis.start:
        lo, hi = vis.start, vis.end
    else:
        lo, hi = np.percentile(plane[::8, ::8], [1, 99.7])
    return np.clip((plane - lo) / max(hi - lo, 1e-6), 0, 1)

labels = [ch.label for ch in channels]
colors = [hex_to_rgb(ch.channel_visualisation.color) for ch in channels]
norm   = [normalise(p, ch) for p, ch in zip(planes, channels)]

views  = [n[..., None] * c for n, c in zip(norm, colors)]
merged = np.clip(sum(views), 0, 1)
views += [merged]
titles = labels + ["merged"]

# --- figure ------------------------------------------------------------------
h, w, panel = planes.shape[1], planes.shape[2], 5.0
fig, axs = plt.subplots(1, len(views), facecolor="#0E1116",
                        figsize=(panel * len(views), panel * h / w + 0.8))

for ax, img, title in zip(np.atleast_1d(axs), views, titles):
    ax.imshow(img, interpolation="nearest")
    ax.set_title(title, color="#C9D1D9", fontsize=11, pad=6)
    ax.axis("off")

fig.suptitle(f"well {row}{col}", color="#E6EDF3", fontsize=14, y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(out_dir / f"{well_name}.png", dpi=300,
            facecolor="#0E1116", bbox_inches="tight")


In [ ]:
# full-pixel-resolution versions of each panel
for img, title in zip(views, titles):
    plt.imsave(out_dir / f"{well_name}_{title}.png", img)